# Pipeline de datos para MIA — paginas web → dataset etiquetado

Toma una lista de paginas web, **baja el HTML** a la carpeta `Raw Texts/`, y con las
funciones de `text_pipeline.py` las **limpia**, las **chunkea** y guarda un **dataset
etiquetado** (member / non-member) en `Datasets/`, listo para enchufarle SAGE / DE-COP.

**Requisitos:** tener `text_pipeline.py` en la misma carpeta que este notebook.

**Formato de salida** (por cada largo en `TARGETS`): `Datasets/dataset_lenN.csv` y `.jsonl`, con
columnas `text, label, title, source, url, date, n_words, n_tokens, length_bucket`.
`label` = 1 (miembro / en training) o 0 (no-miembro). Si algun metodo espera el texto en un
campo llamado `input` (como WikiMIA), renombralo al cargar.


## 1. Instalación

In [2]:
%pip install -q trafilatura pysbd pandas requests tiktoken

Note: you may need to restart the kernel to use updated packages.


## 2. Configuración

`PAGES` es la lista que se puede extender con cuantas paginas se quiera. Cada entrada:
`url` (obligatorio), `label` (1=miembro, 0=no-miembro), y opcionalmente `title` y `date`.
Si se deja `label=None` y se pone `date`, se **deriva** del `CUTOFF` (anterior al cutoff = miembro).


In [3]:
# ===================== CONFIG — editá libremente =====================
PAGES = [
    # --- MIEMBROS: obras de dominio publico, todos los modelos las vieron ---
    {"url": "https://www.gutenberg.org/cache/epub/98/pg98-images.html",
     "title": "A Tale of Two Cities", "label": 1, "date": "1859"},
    {"url": "https://www.gutenberg.org/cache/epub/730/pg730-images.html",
     "title": "Oliver Twist",          "label": 1, "date": "1838"},
    {"url": "https://www.gutenberg.org/cache/epub/1400/pg1400-images.html",
     "title": "Great Expectations",    "label": 1, "date": "1861"},

    # --- NO-MIEMBROS: texto posterior al cutoff del modelo (ejemplo) ---
    {"url": "https://en.wikipedia.org/wiki/2025_in_video_games",
     "title": "2025 in video games",   "label": 0, "date": "2025"},

]

RAW_DIR  = "Raw Texts"      # HTML crudo descargado
OUT_DIR  = "Datasets"       # datasets finales (chunks + label)
TARGETS  = [128]            # largos a generar. Para los 3: [64, 128, 256]
CUTOFF   = "2023"           # si label=None, date < CUTOFF -> miembro
MAX_CHUNKS_PER_PAGE = 50  # None = todos. Poné p.ej. 50 para un test rapido
USER_AGENT = "Mozilla/5.0 (research project)"
# =====================================================================
print(f"{len(PAGES)} paginas configuradas")

4 paginas configuradas


## 3. Imports + contador de tokens

In [4]:
import os, re, json, time
import requests
import pandas as pd
from text_pipeline import build_chunk_dataset, est_token_count

# Contador de largo: intenta tiktoken (tokens reales); si no hay red, estima por palabras.
# Es tu "regla" unica para TODOS los modelos -> mantené el mismo siempre.
try:
    from text_pipeline import make_tiktoken_counter
    COUNT_FN = make_tiktoken_counter("cl100k_base")
    print("Contador: tiktoken (tokens reales)")
except Exception as e:
    COUNT_FN = est_token_count
    print(f"Contador: estimacion por palabras (tiktoken no disponible: {type(e).__name__})")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

Contador: tiktoken (tokens reales)


## 4. Bajar el HTML a `Raw Texts/`

Cachea: si el archivo ya existe, no lo vuelve a bajar. Los fallos no cortan la corrida.
Deja un `_manifest.csv` con el mapeo archivo ↔ url ↔ label ↔ date.


In [5]:
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")[:80]

def resolve_label(p):
    if p.get("label") is not None:
        return int(p["label"])
    d = str(p.get("date", ""))
    return 1 if (d and d <= str(CUTOFF)) else 0   # anterior al cutoff = miembro

manifest = []
for p in PAGES:
    name  = safe_name(p.get("title") or p["url"])
    path  = os.path.join(RAW_DIR, name + ".html")
    label = resolve_label(p)
    if os.path.exists(path):
        print(f"[cache] {name}")
    else:
        try:
            r = requests.get(p["url"], headers={"User-Agent": USER_AGENT}, timeout=30)
            r.raise_for_status()
            with open(path, "w", encoding="utf-8") as f:
                f.write(r.text)
            print(f"[ok]    {name}  ({len(r.text):,} chars)")
            time.sleep(1)   # cortesia con el servidor
        except Exception as e:
            print(f"[FAIL]  {name}: {e}")
            continue
    manifest.append({"file": path, "url": p["url"],
                     "title": p.get("title") or name,
                     "date": p.get("date"), "label": label})

man_df = pd.DataFrame(manifest)
man_df.to_csv(os.path.join(RAW_DIR, "_manifest.csv"), index=False)
print(f"\n{len(man_df)} paginas listas en '{RAW_DIR}/'")
man_df[["title", "label", "date"]]

[ok]    A_Tale_of_Two_Cities  (943,883 chars)
[ok]    Oliver_Twist  (987,157 chars)
[ok]    Great_Expectations  (1,093,438 chars)
[ok]    2025_in_video_games  (436,739 chars)

4 paginas listas en 'Raw Texts/'


,title,label,date
0,A Tale of Two Cities,1,1859
1,Oliver Twist,1,1838
2,Great Expectations,1,1861
3,2025 in video games,0,2025


## 5. Limpiar + chunkear + etiquetar

Para cada pagina: lee el HTML crudo, `build_chunk_dataset` (limpia con trafilatura + chunkea),
le pega el `label`/`title`/`url`, y al final deduplica chunks identicos a nivel global.


In [6]:
def build_for_target(target):
    parts = []
    for m in manifest:
        with open(m["file"], encoding="utf-8") as f:
            raw = f.read()
        dfp = build_chunk_dataset(
            [{"raw": raw, "source": m["title"], "date": m["date"]}],
            target=target, count_fn=COUNT_FN, dedup=True)
        if MAX_CHUNKS_PER_PAGE:
            dfp = dfp.head(MAX_CHUNKS_PER_PAGE)
        dfp["label"] = m["label"]
        dfp["title"] = m["title"]
        dfp["url"]   = m["url"]
        parts.append(dfp)
    df = pd.concat(parts, ignore_index=True)
    df = df.drop_duplicates(subset="text").reset_index(drop=True)   # dedup global
    return df

## 6. Guardar el dataset en `Datasets/` (CSV + JSONL)

In [8]:
COLS = ["text", "label", "title", "source", "url", "date",
        "n_words", "n_tokens", "length_bucket"]

last_df = None
for target in TARGETS:
    df = build_for_target(target)[COLS]
    base = os.path.join(OUT_DIR, f"dataset_len{target}")
    df.to_csv(base + ".csv", index=False)
    with open(base + ".jsonl", "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")
    n_m = int((df.label == 1).sum()); n_n = int((df.label == 0).sum())
    print(f"len={target:>3}: {len(df):>5} chunks  | miembros={n_m}  no-miembros={n_n}")
    print(f"          -> {base}.csv  +  {base}.jsonl")
    last_df = df

print("\nListo. Datasets en la carpeta 'Datasets/'.")

len=128:   200 chunks  | miembros=150  no-miembros=50
          -> Datasets/dataset_len128.csv  +  Datasets/dataset_len128.jsonl

Listo. Datasets en la carpeta 'Datasets/'.


## 7. Vistazo rapido

Chequea el balance member/non-member y la distribucion de largo (acordate de igualar
la distribucion de `n_tokens` entre clases antes de calcular AUC, para que el largo no
filtre la label).


In [9]:
print("Por label:")
print(last_df.groupby("label").size(), "\n")
print("n_tokens por label (describe):")
print(last_df.groupby("label")["n_tokens"].describe()[["count","mean","min","max"]], "\n")
last_df.head()

Por label:
label
0     50
1    150
dtype: int64 

n_tokens por label (describe):
       count    mean   min     max
label                             
0       50.0  169.44  33.0  1090.0
1      150.0  106.02  27.0   328.0 



,text,label,title,source,url,date,n_words,n_tokens,length_bucket
0,"It was the best of times, it was the worst of ...",1,A Tale of Two Cities,A Tale of Two Cities,https://www.gutenberg.org/cache/epub/98/pg98-i...,1859,118,144,128
1,There were a king with a large jaw and a queen...,1,A Tale of Two Cities,A Tale of Two Cities,https://www.gutenberg.org/cache/epub/98/pg98-i...,1859,93,104,128
2,Mrs. Southcott had recently attained her five-...,1,A Tale of Two Cities,A Tale of Two Cities,https://www.gutenberg.org/cache/epub/98/pg98-i...,1859,71,92,64
3,Mere messages in the earthly order of events h...,1,A Tale of Two Cities,A Tale of Two Cities,https://www.gutenberg.org/cache/epub/98/pg98-i...,1859,81,96,64
4,"Under the guidance of her Christian pastors, s...",1,A Tale of Two Cities,A Tale of Two Cities,https://www.gutenberg.org/cache/epub/98/pg98-i...,1859,68,79,64
